### Schrebergärten – Lösung Schritt für Schritt

*37. Bundeswettbewerb Informatik (2018), 1. Runde, Aufgabe 4*

Der Vorstand der neu gegründeten Schrebergartensiedlung von Rechteckingen bietet jedem Einwohner an, sich auf einer Wiese vor dem Ort einen Schrebergarten anzulegen. Ein Schrebergarten ist dabei immer rechteckig und nach den Himmelsrichtungen ausgerichtet. Jeder Schrebergarten hat also eine Nord-Süd-Ausdehnung (Länge) und eine Ost-West-Ausdehnung (Breite).

Jeder Interessent soll ein Stück Land der von ihm gewünschten Ausmaße seines Schrebergartens zugeteilt bekommen. Dem Eigentümer der Wiese soll dafür ein rechteckiges Grundstück abgekauft werden, in das alle Schrebergärten hineinpassen und dessen Fläche möglichst klein ist. Dazu müssen die Schrebergärten auf der Wiese geeignet nebeneinander angeordnet werden. Zufahrtswege werden hierbei nicht berücksichtigt.

#### Aufgabe 4

Hilf dem Vorstand und schreibe ein Programm, welches für eine gegebene Menge von Schrebergärten (jeweils bestimmt durch ihre Länge und Breite) eine möglichst günstige Anordnung auf der Wiese berechnet und grafisch ausgibt. Du kannst davon ausgehen, dass alle Längen und Breiten der Schrebergärten natürliche Zahlen sind.

Wende dein Programm auf die Beispiele an, die du auf den BWINF-Webseiten findest.

#### Dateien (`beispieldaten/`)

`schrebergaerten.txt` enthält drei Beispiele (jeder Garten als `Länge x Breite`, Drehungen werden nicht benötigt):

1. 25 x 15, 15 x 30, 15 x 25, 25 x 20 (das Beispiel der Aufgabe)
2. 6 x 3, 2 x 2, 3 x 1, 4 x 4, 4 x 4
3. 4 x 4, 2 x 3, 6 x 1, 5 x 2, 3 x 5

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Untere Schranke:** Das Grundstück kann nicht kleiner sein als die Summe aller Gartenflächen. Es muss außerdem mindestens so lang sein wie der längste und mindestens so breit wie der breiteste Garten.
- **Kandidaten der Reihe nach prüfen:** Wir zählen alle möglichen Grundstücke (Länge × Breite) auf, sortieren sie nach der Fläche und probieren sie in dieser Reihenfolge. Das erste, in das alle Gärten passen, ist **optimal**. Ein Greedy-Verfahren, das nur ein Grundstück baut, wäre dagegen nur eine Näherung.
- **Backtracking:** Wir setzen einen Garten nach dem anderen. Geht es nicht weiter, nehmen wir den letzten Garten wieder weg und probieren eine andere Möglichkeit.
- **Skyline (Höhenprofil):** Statt jedes Feld einzeln zu verwalten, merken wir uns für jede Spalte, wie weit sie von oben schon gefüllt ist. Das ist eine Liste von Zahlen.
- **Rekursion:** Die Funktion, die den nächsten Garten setzt, ruft sich selbst für den übernächsten auf.

**Nützliche Funktionen**
- `min(liste)`, `liste.index(wert)` (Position des ersten Vorkommens), `sorted(...)` mit Tupeln: Tupel werden zuerst nach dem ersten Eintrag sortiert, bei Gleichstand nach dem zweiten.
- Mengen (`set`), um gleich große Gärten nicht doppelt zu probieren.

## Die Idee

1. Wir rechnen die **Summe der Flächen** aus. Das ist die kleinste mögliche Fläche des Grundstücks.
2. Wir erzeugen alle Grundstücke `H × B` (Länge `H` in Nord-Süd-Richtung, Breite `B` in Ost-West-Richtung), die mindestens so groß wie diese Summe sind, und sortieren sie nach der Fläche.
3. Für jedes Grundstück, beginnend mit dem kleinsten, fragen wir: **Passen alle Gärten hinein?** Dazu setzen wir die Gärten mit Backtracking. Sobald ein Grundstück passt, sind wir fertig.

Weil wir **jede Anordnung**, die überhaupt in Frage kommt, durchsuchen, bevor wir ein Grundstück verwerfen, ist das Ergebnis tatsächlich das kleinstmögliche. Die Lösungshinweise stellen ein schnelleres Näherungsverfahren vor (Garten für Garten an die erste freie Stelle ganz links setzen, ohne Zurücknehmen). Wir suchen dagegen gründlich; bei den kleinen Beispielen geht das schnell.

## Schritt 1: Beispiele einlesen

In der Datei steht jedes Beispiel nach einer Zeile wie `1. Beispiel:` in der nächsten Zeile. Jeder Garten ist `Länge x Breite`, getrennt durch Kommas. Wir speichern ein Beispiel als Liste von Paaren `(laenge, breite)`.

In [ ]:
def lies_beispiele(dateiname):
    """Liest die Beispiele und gibt eine Liste von Beispielen zurück (jedes eine Liste von Paaren (laenge, breite))."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = [z.strip() for z in datei.read().splitlines()]
    beispiele = []
    for i in range(len(zeilen)):
        if 'Beispiel:' in zeilen[i]:
            gaerten = []
            for teil in zeilen[i + 1].strip('.').split(','):
                laenge, breite = teil.split('x')
                gaerten.append((int(laenge), int(breite)))
            beispiele.append(gaerten)
    return beispiele


beispiele = lies_beispiele('beispieldaten/schrebergaerten.txt')
for nr, gaerten in enumerate(beispiele, 1):
    print(nr, gaerten)

1 [(25, 15), (15, 30), (15, 25), (25, 20)]
2 [(6, 3), (2, 2), (3, 1), (4, 4), (4, 4)]
3 [(4, 4), (2, 3), (6, 1), (5, 2), (3, 5)]


## Schritt 2: Einfache Grenzen

Zwei Anordnungen gehen immer: alle Gärten **nebeneinander** (Länge = längster Garten, Breite = Summe der Breiten) oder **übereinander** (Länge = Summe der Längen, Breite = breitester Garten). Das gibt eine obere Schranke für die gesuchte Fläche. Die untere Schranke ist die Summe der Gartenflächen.

In [ ]:
def summe_flaechen(gaerten):
    summe = 0
    for laenge, breite in gaerten:
        summe = summe + laenge * breite
    return summe


def einfache_grenzen(gaerten):
    """Gibt (untere Schranke, Fläche nebeneinander, Fläche übereinander) zurück."""
    laengen = [laenge for laenge, breite in gaerten]
    breiten = [breite for laenge, breite in gaerten]
    nebeneinander = max(laengen) * sum(breiten)
    uebereinander = sum(laengen) * max(breiten)
    return summe_flaechen(gaerten), nebeneinander, uebereinander


for nr, gaerten in enumerate(beispiele, 1):
    print('Beispiel', nr, '(untere Schranke, nebeneinander, übereinander):', einfache_grenzen(gaerten))

Beispiel 1 (untere Schranke, nebeneinander, übereinander): (1700, 2250, 2400)
Beispiel 2 (untere Schranke, nebeneinander, übereinander): (57, 84, 76)
Beispiel 3 (untere Schranke, nebeneinander, übereinander): (53, 90, 100)


Die gesuchte Fläche liegt bei jedem Beispiel zwischen der Summe der Flächen und dem kleineren der beiden anderen Werte.

## Schritt 3: Passen alle Gärten in ein gegebenes Grundstück?

Wir füllen das Grundstück **von oben nach unten**. Für jede der `B` Spalten merken wir in der Liste `hoehe`, wie viele Zeilen von oben schon belegt sind. Am Anfang ist alles `0`.

In jedem Schritt suchen wir die Stelle, die am **weitesten oben** liegt, und davon die am weitesten links: die erste Spalte `s`, in der `hoehe` am kleinsten ist. Dort gibt es eine Lücke, die `L` Spalten breit ist (alle Spalten rechts von `s` mit gleicher Höhe). Für diese Lücke gibt es nur zwei Möglichkeiten:

1. **Ein noch nicht gesetzter Garten** wird mit seiner linken oberen Ecke in diese Stelle gesetzt, wenn er höchstens `L` breit ist und nicht unten aus dem Grundstück ragt. Dann steigt `hoehe` in seinen Spalten um seine Länge.
2. **Die Lücke bleibt leer** (Verschnitt). Dann heben wir sie auf die Höhe des niedrigeren Nachbarn an; der Verschnitt ist die Fläche der Lücke bis dahin. Das darf nur sein, solange der Verschnitt insgesamt nicht größer wird als das, was das Grundstück an Platz über die Summe der Gartenflächen hinaus hat.

Das deckt wirklich **alle** Anordnungen ab: Die Stelle ganz oben links muss entweder die linke obere Ecke eines Gartens sein oder frei bleiben. Gesetzt werden die Gärten also in der Reihenfolge von oben links nach unten rechts, nicht in der Reihenfolge der Liste. Haben zwei Gärten dieselben Maße, probieren wir nur einen von beiden (die schon probierten Maße stehen in einer Menge).

Sind alle Gärten gesetzt, passt es. Zusätzlich zählen wir in `aufrufe`, wie oft die Funktion `setze` aufgerufen wird, um die Arbeit zu messen.

In [ ]:
aufrufe = [0]       # zählt die Aufrufe von setze (eine Liste, damit die Funktion den Wert ändern darf)


def setze(gaerten, benutzt, hoehe, laenge_gesamt, verschnitt_rest, plan):
    """Setzt die noch nicht benutzten Gärten ins Grundstück (laenge_gesamt Zeilen, len(hoehe) Spalten).
    benutzt[i] sagt, ob Garten i schon gesetzt ist; plan[i] ist (Zeile, Spalte) seiner linken oberen Ecke.
    Gibt True zurück, wenn alle Gärten untergebracht werden konnten."""
    aufrufe[0] = aufrufe[0] + 1
    if False not in benutzt:
        return True
    breite_gesamt = len(hoehe)
    # Stelle ganz oben, davon ganz links, und die Breite der Lücke dort
    s = hoehe.index(min(hoehe))
    zeile = hoehe[s]
    luecke = 1
    while s + luecke < breite_gesamt and hoehe[s + luecke] == zeile:
        luecke = luecke + 1

    # Möglichkeit 1: einen noch nicht gesetzten Garten hierher setzen
    probiert = set()
    for i in range(len(gaerten)):
        laenge, breite = gaerten[i]
        if benutzt[i] or gaerten[i] in probiert:
            continue
        if breite > luecke or zeile + laenge > laenge_gesamt:
            continue
        probiert.add(gaerten[i])
        benutzt[i] = True
        plan[i] = (zeile, s)
        for spalte in range(s, s + breite):
            hoehe[spalte] = hoehe[spalte] + laenge
        if setze(gaerten, benutzt, hoehe, laenge_gesamt, verschnitt_rest, plan):
            return True
        for spalte in range(s, s + breite):          # zurücknehmen
            hoehe[spalte] = hoehe[spalte] - laenge
        benutzt[i] = False

    # Möglichkeit 2: Lücke leer lassen (auf die Höhe des niedrigeren Nachbarn anheben)
    nachbarn = []
    if s > 0:
        nachbarn.append(hoehe[s - 1])
    if s + luecke < breite_gesamt:
        nachbarn.append(hoehe[s + luecke])
    if len(nachbarn) > 0:
        ziel = min(nachbarn)
        kosten = luecke * (ziel - zeile)
        if kosten <= verschnitt_rest:
            for spalte in range(s, s + luecke):
                hoehe[spalte] = ziel
            if setze(gaerten, benutzt, hoehe, laenge_gesamt, verschnitt_rest - kosten, plan):
                return True
            for spalte in range(s, s + luecke):
                hoehe[spalte] = zeile
    return False


def passt(gaerten, laenge_gesamt, breite_gesamt):
    """Gibt den Plan (Liste von (Zeile, Spalte) je Garten) zurück, wenn alle Gärten ins Grundstück passen, sonst None."""
    verschnitt = laenge_gesamt * breite_gesamt - summe_flaechen(gaerten)
    benutzt = [False] * len(gaerten)
    plan = [None] * len(gaerten)
    hoehe = [0] * breite_gesamt
    if setze(gaerten, benutzt, hoehe, laenge_gesamt, verschnitt, plan):
        return plan
    return None


# Beispiel 2 (Fläche 57): passt es in 9 x 8 (72)? In 7 x 8 (56)?
aufrufe[0] = 0
print(passt(beispiele[1], 9, 8), aufrufe[0], 'Aufrufe')
print(passt(beispiele[1], 7, 8))

[(0, 0), (6, 0), (0, 3), (0, 4), (4, 3)] 34 Aufrufe
None


Die Ausgabe gibt für jeden Garten die Ecke `(Zeile, Spalte)` an (in der Reihenfolge der Eingabe). Ein Grundstück mit 56 Quadratmetern ist kleiner als die Summe der Flächen (57), also gibt es dort keinen Plan (`None`).

## Schritt 4: Das kleinste Grundstück suchen

Jetzt erzeugen wir die Kandidaten. Länge und Breite des Grundstücks reichen jeweils vom größten Maß eines Gartens bis zur Summe aller Maße (länger oder breiter braucht man nie). Wir behalten nur die Kandidaten mit Fläche mindestens der Summe der Gartenflächen, sortieren die Tupel `(Fläche, Länge, Breite)` aufsteigend und probieren sie der Reihe nach.

In [ ]:
def loese(gaerten):
    """Gibt (Fläche, Länge, Breite, Plan) des kleinsten Grundstücks zurück, in das alle Gärten passen."""
    mindest = summe_flaechen(gaerten)
    laengen = [laenge for laenge, breite in gaerten]
    breiten = [breite for laenge, breite in gaerten]
    kandidaten = []
    for h in range(max(laengen), sum(laengen) + 1):
        for b in range(max(breiten), sum(breiten) + 1):
            if h * b >= mindest:
                kandidaten.append((h * b, h, b))
    kandidaten.sort()
    for flaeche, h, b in kandidaten:
        plan = passt(gaerten, h, b)
        if plan is not None:
            return flaeche, h, b, plan


for nr, gaerten in enumerate(beispiele, 1):
    aufrufe[0] = 0
    flaeche, h, b, plan = loese(gaerten)
    print('Beispiel', nr, ': Fläche', flaeche, '=', h, 'x', b, '(Summe der Gartenflächen:', summe_flaechen(gaerten), ',', aufrufe[0], 'Aufrufe von setze)')

Beispiel 1 : Fläche 1800 = 40 x 45 (Summe der Gartenflächen: 1700 , 1550 Aufrufe von setze)
Beispiel 2 : Fläche 63 = 9 x 7 (Summe der Gartenflächen: 57 , 350 Aufrufe von setze)
Beispiel 3 : Fläche 56 = 7 x 8 (Summe der Gartenflächen: 53 , 172 Aufrufe von setze)


Dass es **nicht kleiner** geht, folgt daraus, dass alle kleineren Kandidaten vollständig durchgespielt und verworfen wurden. Bei Beispiel 1 bleibt ein Verschnitt von 100 (1800 − 1700), bei Beispiel 2 sind es 6 und bei Beispiel 3 sind es 3.

## Schritt 5: Grafische Ausgabe

Die Aufgabe verlangt eine Zeichnung. Mit Textzeichen geht das ohne Zusatzpaket: Jeder Garten bekommt einen Buchstaben (`A`, `B`, ...), und wir füllen damit ein Gitter aus Zeilen und Spalten; `.` steht für Verschnitt. Das Gitter ist so groß wie das Grundstück (1 Zeichen je Quadratmeter). Die Funktion prüft nebenbei, dass sich nie zwei Gärten überdecken und kein Garten über den Rand ragt.

In [ ]:
def zeichne(gaerten, h, b, plan):
    """Gibt das Grundstück als Text aus: Buchstaben für Gärten, Punkt für Verschnitt."""
    gitter = []
    for zeile in range(h):
        gitter.append(['.'] * b)
    for i in range(len(gaerten)):
        laenge, breite = gaerten[i]
        zeile0, spalte0 = plan[i]
        buchstabe = chr(ord('A') + i)
        assert zeile0 + laenge <= h and spalte0 + breite <= b
        for zeile in range(zeile0, zeile0 + laenge):
            for spalte in range(spalte0, spalte0 + breite):
                assert gitter[zeile][spalte] == '.'        # nicht überlappt
                gitter[zeile][spalte] = buchstabe
    for zeile in gitter:
        print(''.join(zeile))


def zeige(nr):
    gaerten = beispiele[nr - 1]
    flaeche, h, b, plan = loese(gaerten)
    for i in range(len(gaerten)):
        print(chr(ord('A') + i), ': Länge', gaerten[i][0], 'x Breite', gaerten[i][1], ', linke obere Ecke (Zeile, Spalte) =', plan[i])
    print('Grundstück:', h, 'x', b, '=', flaeche)
    zeichne(gaerten, h, b, plan)


zeige(2)
print()
zeige(3)

A : Länge 6 x Breite 3 , linke obere Ecke (Zeile, Spalte) = (0, 0)
B : Länge 2 x Breite 2 , linke obere Ecke (Zeile, Spalte) = (6, 0)
C : Länge 3 x Breite 1 , linke obere Ecke (Zeile, Spalte) = (6, 2)
D : Länge 4 x Breite 4 , linke obere Ecke (Zeile, Spalte) = (0, 3)
E : Länge 4 x Breite 4 , linke obere Ecke (Zeile, Spalte) = (4, 3)
Grundstück: 9 x 7 = 63
AAADDDD
AAADDDD
AAADDDD
AAADDDD
AAAEEEE
AAAEEEE
BBCEEEE
BBCEEEE
..C....

A : Länge 4 x Breite 4 , linke obere Ecke (Zeile, Spalte) = (0, 3)
B : Länge 2 x Breite 3 , linke obere Ecke (Zeile, Spalte) = (0, 0)
C : Länge 6 x Breite 1 , linke obere Ecke (Zeile, Spalte) = (0, 7)
D : Länge 5 x Breite 2 , linke obere Ecke (Zeile, Spalte) = (2, 0)
E : Länge 3 x Breite 5 , linke obere Ecke (Zeile, Spalte) = (4, 2)
Grundstück: 7 x 8 = 56
BBBAAAAC
BBBAAAAC
DD.AAAAC
DD.AAAAC
DDEEEEEC
DDEEEEEC
DDEEEEE.


Die Buchstaben stehen in der Reihenfolge der Eingabe. `.` ist Verschnitt.

In [ ]:
zeige(1)

A : Länge 25 x Breite 15 , linke obere Ecke (Zeile, Spalte) = (0, 30)
B : Länge 15 x Breite 30 , linke obere Ecke (Zeile, Spalte) = (0, 0)
C : Länge 15 x Breite 25 , linke obere Ecke (Zeile, Spalte) = (25, 20)
D : Länge 25 x Breite 20 , linke obere Ecke (Zeile, Spalte) = (15, 0)
Grundstück: 40 x 45 = 1800
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBAAAAAAAAAAAAAAA
DDD

## Für Neugierige: wie weit kommt das Verfahren?

Die Lösungshinweise erwähnen als Maßstab das Packen **aufsteigender Quadrate** `1 × 1, 2 × 2, ..., n × n` (eine Tabelle optimaler Lösungen aus einer Veröffentlichung von Richard Korf, in den Lösungshinweisen nur als Bild). Wir probieren das Verfahren bis `n = 8` aus. Mit dem Zählen der Aufrufe sieht man, wie stark die Arbeit wächst; ab `n = 9` dauert die Suche schon mehrere Sekunden (in JupyterLite länger).

In [ ]:
for n in range(1, 9):
    quadrate = [(k, k) for k in range(1, n + 1)]
    aufrufe[0] = 0
    flaeche, h, b, plan = loese(quadrate)
    print('n =', n, ':', h, 'x', b, '=', flaeche, '(Summe der Flächen:', summe_flaechen(quadrate), ',', aufrufe[0], 'Aufrufe)')

n = 1 : 1 x 1 = 1 (Summe der Flächen: 1 , 2 Aufrufe)
n = 2 : 2 x 3 = 6 (Summe der Flächen: 5 , 3 Aufrufe)
n = 3 : 3 x 5 = 15 (Summe der Flächen: 14 , 8 Aufrufe)
n = 4 : 5 x 7 = 35 (Summe der Flächen: 30 , 120 Aufrufe)
n = 5 : 5 x 12 = 60 (Summe der Flächen: 55 , 307 Aufrufe)
n = 6 : 9 x 11 = 99 (Summe der Flächen: 91 , 5064 Aufrufe)
n = 7 : 7 x 22 = 154 (Summe der Flächen: 140 , 58369 Aufrufe)
n = 8 : 14 x 15 = 210 (Summe der Flächen: 204 , 84823 Aufrufe)


- Die Zahl der Aufrufe wächst schnell: Die Aufgabe ist NP-schwer, im schlechtesten Fall müssen alle Anordnungen durchprobiert werden.
- Schneller wird das Verfahren, wenn man die Kandidaten klüger wählt oder bessere untere Schranken benutzt (z. B. Verschnitt, der in Lücken sicher entsteht, früher abziehen). Das Näherungsverfahren der Lösungshinweise setzt die Gärten nur einmal, absteigend nach der Höhe sortiert, und kommt dadurch mit sehr vielen Gärten zurecht, liefert aber nicht immer das Optimum.
- Man kann auch mehrere zufällige Reihenfolgen mit einem Greedy-Verfahren probieren und die beste Lösung behalten.

## Zum Schluss: Dasselbe mit MiniZinc

Die Aufgabe passt gut zu MiniZinc: Wir beschreiben nur, **was** gelten soll, und der Solver sucht. Jeder Garten bekommt als Unbekannte die Zeile `z[i]` und die Spalte `s[i]` seiner linken oberen Ecke; auch die Maße `H` und `W` des Grundstücks sind Unbekannte. Die Bedingungen:

- Jeder Garten liegt ganz im Grundstück: `z[i] + laenge[i] <= H` und `s[i] + breite[i] <= W`.
- Keine zwei Gärten überlappen sich. Dafür gibt es die fertige Bedingung `diffn(z, s, laenge, breite)` aus `diffn.mzn`: Je zwei Rechtecke müssen sich in mindestens einer Richtung ausschließen (eines liegt ganz links, rechts, über oder unter dem anderen).
- Ziel: `minimize H * W`, also die Fläche des Grundstücks.

Die Modelle sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den [MiniZinc Playground](https://play.minizinc.org/)).

Das Modell (`schrebergaerten.mzn`):

```minizinc
include "diffn.mzn";
int: n;                                   % Anzahl der Gärten
array[1..n] of int: laenge;               % Nord-Süd-Ausdehnung
array[1..n] of int: breite;               % Ost-West-Ausdehnung

var max(laenge)..sum(laenge): H;          % Länge des Grundstücks
var max(breite)..sum(breite): W;          % Breite des Grundstücks
array[1..n] of var 0..sum(laenge): z;     % Zeile der linken oberen Ecke
array[1..n] of var 0..sum(breite): s;     % Spalte der linken oberen Ecke

% jeder Garten liegt ganz im Grundstück
constraint forall(i in 1..n)(z[i] + laenge[i] <= H /\ s[i] + breite[i] <= W);
% keine zwei Gärten überlappen sich
constraint diffn(z, s, laenge, breite);
% (überflüssig, aber hilft dem Solver) Das Grundstück ist mindestens so groß wie alle Gärten zusammen
constraint H * W >= sum(i in 1..n)(laenge[i] * breite[i]);

solve minimize H * W;
output ["Fläche \(H * W) = \(H) x \(W)\n", "z = \(z)\n", "s = \(s)\n"];
```

Die Daten schreiben wir mit einer Python-Zelle aus den Beispielen in die Form, die MiniZinc erwartet (`.dzn`):

In [ ]:
def als_dzn(gaerten):
    """Beispiel als MiniZinc-Daten (.dzn)."""
    laengen = [laenge for laenge, breite in gaerten]
    breiten = [breite for laenge, breite in gaerten]
    return 'n = ' + str(len(gaerten)) + ';\nlaenge = ' + str(laengen) + ';\nbreite = ' + str(breiten) + ';'


print(als_dzn(beispiele[1]))

n = 5;
laenge = [6, 2, 3, 4, 4];
breite = [3, 2, 1, 4, 4];


Ergebnis für Beispiel 1 (Solver Gecode, bei jedem Beispiel weniger als eine Sekunde):

**Ausgabe:**

```
Fläche 1800 = 40 x 45
z = [15, 25, 0, 0]
s = [30, 0, 20, 0]
----------
==========
```

Beispiel 2:

**Ausgabe:**

```
Fläche 63 = 9 x 7
z = [0, 6, 6, 4, 0]
s = [4, 4, 6, 0, 0]
----------
==========
```

Beispiel 3:

**Ausgabe:**

```
Fläche 56 = 7 x 8
z = [3, 5, 0, 0, 0]
s = [0, 4, 7, 5, 0]
----------
==========
```

Der Solver beweist die Optimalität selbst. Die Flächen stimmen mit unserem Programm überein; die Anordnungen können sich unterscheiden, weil es mehrere gleich gute gibt. Bei vielen oder sehr großen Gärten wird der Beweis, dass es nicht kleiner geht, auch für den Solver schwer.